In [4]:
import numpy as np
import copy
np.set_printoptions(suppress=True)  # 禁用科学计数
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

In [2]:
class Node:
    """决策树节点"""
    def __init__(self):
        self.is_leaf = False
        self.label = None          # 如果是叶子节点，存类别
        self.feature_idx = None    # 分裂特征索引
        self.split_value = None    # 数值特征：阈值；分类型：None
        self.children = {}         # 分类型：{取值: 子节点}；数值型：{'left': 子节点, 'right': 子节点}
        self.feature_type = None   # 'numeric' 或 'category'

In [3]:
class DecisionTree:
    def __init__(self, criterion='entropy', max_depth=None, min_samples_split=2):
        self.criterion = criterion          # 'entropy' 或 'gini'
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None
        self.feature_types = []             # 记录每个特征是数值型还是分类型
        self.feature_names = None

    def _calc_entropy(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probs = counts / len(y)
        return -np.sum(probs[probs > 0] * np.log2(probs[probs > 0]))

    def _calc_gini(self, y):
        if len(y) == 0:
            return 0
        counts = np.bincount(y)
        probs = counts / len(y)
        return 1 - np.sum(probs ** 2)

    def _calc_impurity(self, y):
        if self.criterion == 'entropy':
            return self._calc_entropy(y)
        else:
            return self._calc_gini(y)

    def _info_gain(self, y, y_left, y_right):
        """数值型特征的信息增益（二路分割）"""
        p = len(y_left) / len(y)
        impurity_parent = self._calc_impurity(y)
        impurity_children = p * self._calc_impurity(y_left) + (1 - p) * self._calc_impurity(y_right)
        return impurity_parent - impurity_children

    def _info_gain_category(self, y, y_groups):
        """分类型特征的信息增益（多路分割）"""
        impurity_parent = self._calc_impurity(y)
        impurity_children = 0
        for group in y_groups:
            if len(group) == 0:
                continue
            impurity_children += (len(group) / len(y)) * self._calc_impurity(group)
        return impurity_parent - impurity_children

    def _best_split(self, X, y, depth):
        """寻找最佳分裂特征和阈值"""
        n_samples, n_features = X.shape
        if n_samples < self.min_samples_split or (self.max_depth and depth >= self.max_depth):
            return None, None, None

        best_gain = -1
        best_feature = None
        best_split = None
        best_groups = None

        for feat_idx in range(n_features):
            feature = X[:, feat_idx]
            unique_vals = np.unique(feature)

            if len(unique_vals) <= 1:
                continue

            # 判断特征类型
            if self.feature_types[feat_idx] == 'category':
                # 分类型：多路分割
                groups = []
                y_groups = []
                for val in unique_vals:
                    mask = (feature == val)
                    groups.append(mask)
                    y_groups.append(y[mask])
                gain = self._info_gain_category(y, y_groups)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feat_idx
                    best_split = None  # 分类型不需要阈值
                    best_groups = groups
            else:
                # 数值型：尝试每个二分点
                sorted_vals = np.sort(unique_vals)
                for i in range(len(sorted_vals) - 1):
                    threshold = (sorted_vals[i] + sorted_vals[i + 1]) / 2
                    mask_left = (feature <= threshold)
                    y_left = y[mask_left]
                    y_right = y[~mask_left]
                    if len(y_left) == 0 or len(y_right) == 0:
                        continue
                    gain = self._info_gain(y, y_left, y_right)
                    if gain > best_gain:
                        best_gain = gain
                        best_feature = feat_idx
                        best_split = threshold
                        best_groups = None

        return best_feature, best_split, best_groups

    def _build_tree(self, X, y, depth):
        """递归构建决策树"""
        node = Node()
        unique_labels = np.unique(y)

        # 如果全部同类，直接返回叶子
        if len(unique_labels) == 1:
            node.is_leaf = True
            node.label = unique_labels[0]
            return node

        # 如果无法继续分裂，返回多数类叶子
        if len(y) < self.min_samples_split or (self.max_depth and depth >= self.max_depth):
            node.is_leaf = True
            node.label = Counter(y).most_common(1)[0][0]
            return node

        best_feature, best_split, best_groups = self._best_split(X, y, depth)

        # 找不到有效分裂，返回多数类叶子
        if best_feature is None:
            node.is_leaf = True
            node.label = Counter(y).most_common(1)[0][0]
            return node

        node.feature_idx = best_feature
        node.split_value = best_split
        node.feature_type = self.feature_types[best_feature]

        if self.feature_types[best_feature] == 'category':
            # 分类型：按每个取值分裂
            feature = X[:, best_feature]
            unique_vals = np.unique(feature)
            for i, val in enumerate(unique_vals):
                mask = best_groups[i]
                child = self._build_tree(X[mask], y[mask], depth + 1)
                node.children[val] = child
        else:
            # 数值型：左右分裂
            mask_left = (X[:, best_feature] <= best_split)
            child_left = self._build_tree(X[mask_left], y[mask_left], depth + 1)
            child_right = self._build_tree(X[~mask_left], y[~mask_left], depth + 1)
            node.children['left'] = child_left
            node.children['right'] = child_right

        return node

    def _post_prune(self, node, X_val, y_val):
        """后剪枝：自底向上考察，如果替换为叶子能提升验证准确率则剪枝"""
        if node.is_leaf:
            return

        # 递归剪枝子节点
        for key in list(node.children.keys()):
            child = node.children[key]
            # 需要获取该子节点对应的验证集子集
            if node.feature_type == 'category':
                mask = (X_val[:, node.feature_idx] == key)
            else:
                if key == 'left':
                    mask = (X_val[:, node.feature_idx] <= node.split_value)
                else:
                    mask = (X_val[:, node.feature_idx] > node.split_value)
            X_sub = X_val[mask]
            y_sub = y_val[mask]
            if len(X_sub) > 0:
                self._post_prune(child, X_sub, y_sub)

        # 检查当前节点是否应该剪枝（变成叶子）
        if not node.is_leaf:
            # 计算剪枝前的验证准确率
            y_pred_before = self._predict_node(node, X_val)
            acc_before = np.mean(y_pred_before == y_val)

            # 计算剪枝后的验证准确率（当前节点的多数类）
            labels_leaf = self._get_node_labels(node)
            leaf_label = Counter(labels_leaf).most_common(1)[0][0] if len(labels_leaf) > 0 else Counter(y_val).most_common(1)[0][0]
            acc_after = np.mean(np.full(len(y_val), leaf_label) == y_val)

            # 如果剪枝后准确率不低于剪枝前，执行剪枝
            if acc_after >= acc_before:
                node.is_leaf = True
                node.label = leaf_label
                node.children = {}
                node.feature_idx = None
                node.split_value = None

    def _get_node_labels(self, node):
        """递归获取节点下所有样本的标签（用于统计多数类）"""
        if node.is_leaf:
            return [node.label]
        labels = []
        for child in node.children.values():
            labels.extend(self._get_node_labels(child))
        return labels

    def _predict_node(self, node, X):
        """单个节点上的预测"""
        if node.is_leaf:
            return np.full(X.shape[0], node.label)
        if node.feature_type == 'category':
            # 分类型：按值匹配
            preds = np.zeros(X.shape[0], dtype=int)
            for val, child in node.children.items():
                mask = (X[:, node.feature_idx] == val)
                if np.any(mask):
                    preds[mask] = self._predict_node(child, X[mask])
            return preds
        else:
            # 数值型：左右分支
            mask_left = (X[:, node.feature_idx] <= node.split_value)
            preds = np.zeros(X.shape[0], dtype=int)
            if np.any(mask_left):
                preds[mask_left] = self._predict_node(node.children['left'], X[mask_left])
            if np.any(~mask_left):
                preds[~mask_left] = self._predict_node(node.children['right'], X[~mask_left])
            return preds

    def fit(self, X, y, pruning='none', X_val=None, y_val=None):
        """
        训练决策树
        pruning: 'none' | 'pre' | 'after'
        如果 pruning='pre' 或 'after' 且未提供 X_val/y_val，则自动从训练集中划分20%作为验证集
        """
        # 统一转换为 numpy 数组
        if isinstance(X, pd.DataFrame):
            self.feature_names = X.columns.tolist()
            X = X.values
        if isinstance(y, pd.Series):
            y = y.values
        if isinstance(y, pd.DataFrame):
            y = y.values.ravel()

        # 检测特征类型
        self.feature_types = []
        for i in range(X.shape[1]):
            col = X[:, i]
            if col.dtype == np.object_ or col.dtype == str or col.dtype.type is np.str_:
                self.feature_types.append('category')
            else:
                self.feature_types.append('numeric')

        # 标签编码（处理中文）
        self.label_map = {}
        unique_labels = np.unique(y)
        for i, label in enumerate(unique_labels):
            self.label_map[label] = i
        y_encoded = np.array([self.label_map[lbl] for lbl in y])
        self.inv_label_map = {v: k for k, v in self.label_map.items()}

        # 处理预剪枝和后剪枝需要的验证集
        self.use_pre_pruning = (pruning == 'pre')
        self.use_post_pruning = (pruning == 'after')

        if self.use_pre_pruning or self.use_post_pruning:
            if X_val is None or y_val is None:
                # 自动划分20%作为验证集
                n = len(X)
                idx = np.random.permutation(n)
                split = int(n * 0.8)
                X_train, X_val = X[idx[:split]], X[idx[split:]]
                y_train, y_val = y_encoded[idx[:split]], y_encoded[idx[split:]]
                print(f"自动划分验证集: 训练集 {len(X_train)} 条, 验证集 {len(X_val)} 条")
            else:
                X_train = X
                y_train = y_encoded
                if isinstance(X_val, pd.DataFrame):
                    X_val = X_val.values
                if isinstance(y_val, pd.Series) or isinstance(y_val, pd.DataFrame):
                    y_val = y_val.values.ravel()
                # 对验证集标签也编码
                y_val = np.array([self.label_map[lbl] for lbl in y_val])
        else:
            X_train, y_train = X, y_encoded

        # 构建完整树
        self.root = self._build_tree(X_train, y_train, 0)

        # 后剪枝（必须在完整树构建后）
        if self.use_post_pruning and X_val is not None and len(X_val) > 0:
            self._post_prune(self.root, X_val, y_val)

        # 预剪枝：实际上在 _build_tree 里已经通过 max_depth/min_samples_split 控制了
        # 如果希望预剪枝更严格（基于验证集提前停止），需要修改 _build_tree
        # 但为了方便，我们依赖 max_depth 和 min_samples_split 作为预剪枝
        if self.use_pre_pruning:
            # 重新构建树，使用更严格的参数（已经通过 max_depth/min_samples_split 控制）
            # 如果想实现基于验证集准确率的预剪枝，需要重写 _build_tree 传入验证集
            # 这里我简化处理：用 max_depth=5, min_samples_split=3 作为预剪枝默认
            if self.max_depth is None:
                self.max_depth = 5
            if self.min_samples_split < 3:
                self.min_samples_split = 3
            # 重新构建
            self.root = self._build_tree(X_train, y_train, 0)

    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values
        if len(X.shape) == 1:
            X = X.reshape(1, -1)
        preds_encoded = self._predict_node(self.root, X)
        # 解码回原始标签
        return np.array([self.inv_label_map[p] for p in preds_encoded])

    def accuracy(self, X, y):
        y_pred = self.predict(X)
        if isinstance(y, pd.Series):
            y = y.values
        if isinstance(y, pd.DataFrame):
            y = y.values.ravel()
        return np.mean(y_pred == y)

In [6]:
# 1. 读取数据
path = 'data/西瓜数据集3,1.txt'
data = pd.read_csv(path, encoding='utf-8')  # 或 gbk

# 2. 准备特征和标签
X = data[['Density', 'Sugar content']].copy()
y = data['Good melon'].copy()

# 3. 实例化决策树（不剪枝）
tree = DecisionTree(criterion='entropy', max_depth=None, min_samples_split=2)
tree.fit(X, y, pruning='none')
print(f"无剪枝 训练集准确率: {tree.accuracy(X, y) * 100:.2f}%")

# 4. 预剪枝（自动划分验证集）
tree_pre = DecisionTree(criterion='entropy', max_depth=3, min_samples_split=3)
tree_pre.fit(X, y, pruning='pre')
print(f"预剪枝 训练集准确率: {tree_pre.accuracy(X, y) * 100:.2f}%")

# 5. 后剪枝（自动划分验证集）
tree_after = DecisionTree(criterion='entropy', max_depth=None, min_samples_split=2)
tree_after.fit(X, y, pruning='after')
print(f"后剪枝 训练集准确率: {tree_after.accuracy(X, y) * 100:.2f}%")

# 6. 查看预测结果
print("预测结果:", tree.predict(X))

FileNotFoundError: [Errno 2] No such file or directory: 'data/西瓜数据集3,1.txt'